#### 多臂老虎机的探索与利用

[Executable Code](./two_armed_bandit.py)

目的：最简单的 RL 问题，演示探索 vs. 利用的核心矛盾

核心设计：

- 环境两个臂，中奖概率分别为 0.6（最优）和 0.4，退化为单状态 MDP。

```python
class TwoArmedBandit:

    def __init__(self, prob_a=0.6, prob_b=0.4):
        self.prob_a = prob_a  # Win probability of arm A
        self.prob_b = prob_b  # Win probability of arm B
        self.best_prob = max(prob_a, prob_b)

    def pull(self, arm):
        if arm == 0:
            return 1 if np.random.random() < self.prob_a else 0
        else:
            return 1 if np.random.random() < self.prob_b else 0
```

- 四种策略对比：

随机：纯随机选臂，不做学习，期望奖励 ≈ 0.5

In [ ]:

def strategy_random(bandit, n_steps):
    """
    Strategy 1: random strategy
    Choose arm A or B completely at random at every step, ignoring history.

    This is the most basic baseline strategy. Since it does no learning at
    all, the average reward should be close to the mean of the two arms'
    probabilities: (0.6 + 0.4) / 2 = 0.5
    """
    rewards = []
    for _ in range(n_steps):
        arm = np.random.choice([0, 1])  # Choose randomly with equal probability
        reward = bandit.pull(arm)
        rewards.append(reward)
    return np.array(rewards)

贪婪：始终选当前估计最高的臂，用增量平均更新 `Q[arm] += (reward - Q[arm]) / counts[arm]`。问题：可能锁死在次优臂上（过早收敛）

In [ ]:

def strategy_greedy(bandit, n_steps):
    """
    Strategy 2: greedy strategy
    Always choose the arm with the highest current estimate.

    Problem: since the initial estimates are equal, the first step picks an
    arm at random; if it happens to win, that arm gets pulled forever and the
    other arm is never explored again. This is the classic example of
    "premature convergence".
    """
    rewards = []
    # Q[a] denotes the current estimated expected reward of arm a
    Q = np.zeros(2)       # Initial estimates
    counts = np.zeros(2)  # Number of times each arm has been pulled

    for _ in range(n_steps):
        # Always choose the arm with the highest current estimate (exploit only)
        arm = np.argmax(Q)
        reward = bandit.pull(arm)
        rewards.append(reward)

        # Update that arm's estimate: incremental running average
        counts[arm] += 1
        Q[arm] += (reward - Q[arm]) / counts[arm]

    return np.array(rewards)

ε-greedy：以概率 ε 随机探索，否则选最优臂，平衡了探索与利用

In [ ]:
def strategy_epsilon_greedy(bandit, n_steps, epsilon=0.1):
    """
    Strategy 3: epsilon-greedy strategy (epsilon = 0.1)
    Explore randomly with probability epsilon, otherwise choose the current
    best arm with probability 1-epsilon.

    epsilon = 0.1 means about 10% of the time is spent on random exploration.
    This is the simplest and most common way to resolve the
    "exploration vs. exploitation" tension.
    epsilon too large → too much time wasted on known-bad choices;
    epsilon too small → the optimal arm may never be found.
    """
    rewards = []
    Q = np.zeros(2)
    counts = np.zeros(2)

    for _ in range(n_steps):
        # Explore randomly with probability epsilon, otherwise choose greedily
        if np.random.random() < epsilon:
            arm = np.random.choice([0, 1])  # Explore
        else:
            arm = np.argmax(Q)  # Exploit

        reward = bandit.pull(arm)
        rewards.append(reward)

        counts[arm] += 1
        Q[arm] += (reward - Q[arm]) / counts[arm]

    return np.array(rewards)

UCB：`Q(a) + c * sqrt(ln(t) / N(a))`，用置信上界引导探索——越少被拉的臂，不确定性越大，越容易被尝试

In [ ]:
def strategy_ucb(bandit, n_steps, c=2.0):
    """
    Strategy 4: Upper Confidence Bound strategy (UCB)
    Choose the arm with the highest "estimate + uncertainty bound".

    UCB formula: Q(a) + c * sqrt(ln(t) / N(a))
    - Q(a): the current estimated expected reward of arm a
    - c: parameter controlling the amount of exploration (typically c=2)
    - t: current total step count
    - N(a): number of times arm a has been chosen

    Core idea: if an arm has rarely been chosen (N(a) is small), the estimate
    for it is uncertain, so its uncertainty bound is large -- UCB therefore
    tends to try the arms that are "still uncertain". As the number of trials
    grows, uncertainty shrinks and the strategy naturally shifts toward
    greedy behavior.
    """
    rewards = []
    Q = np.zeros(2)
    counts = np.zeros(2)

    for t in range(1, n_steps + 1):
        # Pull each arm once in the first two steps, so every arm has been tried
        if t <= 2:
            arm = t - 1
        else:
            # Compute the UCB values
            ucb_values = np.zeros(2)
            for a in range(2):
                # Uncertainty bound: fewer pulls means a larger bound
                uncertainty = c * np.sqrt(np.log(t) / counts[a])
                ucb_values[a] = Q[a] + uncertainty
            arm = np.argmax(ucb_values)

        reward = bandit.pull(arm)
        rewards.append(reward)

        counts[arm] += 1
        Q[arm] += (reward - Q[arm]) / counts[arm]

    return np.array(rewards)

#### Bellman 方程的数值验证

[Executable Code](./bellman_equation_verify.py)

目的：用手算和迭代求解两种方式验证 Bellman 期望方程和最优方程的一致性。

核心设计：

- MDP 定义：3 状态、2 动作的确定/随机混合 MDP，用嵌套字典 `P[s][a] = {next_s: prob}` 和 `R[s][a]` 表示转移概率和即时奖励

- 手动求解 Bellman 期望方程 (`manual_bellman_expectation`)：
    - 固定策略 π（均匀随机 0.5/0.5）
    - 建立线性方程组 `V^π(s) = Σ_a π(a|s)[R(s,a) + γΣ P(s'|s,a)V^π(s')]`
    - 用 `np.linalg.solve(A, b)` 精确求解

In [ ]:
def manual_bellman_expectation():

    print("=" * 60)
    print("  Bellman expectation equation -- manual derivation")
    print("=" * 60)
    print()
    print("Given policy: uniform random π(a|s) = 0.5")
    print(f"Discount factor: γ = {GAMMA}")
    print()
    print("System of equations (v0 = V^π(s0), v1 = V^π(s1), v2 = V^π(s2)):")
    print("  v0 = 1.5   + 0.45*v1 + 0.45*v2  ...... (Equation 1)")
    print("  v1 = -0.5  + 0.225*v0 + 0.36*v1 + 0.315*v2  (Equation 2)")
    print("  v2 = 2.0   + 0.45*v1 + 0.45*v2  ...... (Equation 3)")
    print()

    A = np.array([
        [1.0,   -0.45,   -0.45],
        [-0.225, 0.64,   -0.315],
        [0.0,   -0.45,    0.55],
    ])
    b = np.array([1.5, -0.5, 2.0])

    manual_V = np.linalg.solve(A, b)

    print("Solving the linear system by hand gives:")
    for i in range(N_STATES):
        print(f"  V^π(s{i}) = {manual_V[i]:.6f}")
    print()
    return manual_V

- 策略评估迭代 (`policy_evaluation`)：
    - 从 V=0 出发，反复用 Bellman 期望方程更新：`V(s) ← Σ_a π(a|s)[R + γΣP·V]`
    - 收敛到与手算一致的 V^π(s)

In [ ]:
def policy_evaluation(policy, max_iter=1000, tol=1e-8):
    
    V = np.zeros(N_STATES)
    history = [V.copy()]

    for iteration in range(max_iter):
        V_new = np.zeros(N_STATES)

        for s in range(N_STATES):
            # Bellman expectation equation: weighted sum over all actions
            for a in range(N_ACTIONS):
                # π(a|s) * [R(s,a) + γ * Σ P(s'|s,a) * V(s')]
                action_value = R[s][a]
                for next_s, prob in P[s][a].items():
                    action_value += GAMMA * prob * V[next_s]
                V_new[s] += policy[s][a] * action_value

        # Check for convergence
        delta = np.max(np.abs(V_new - V))
        history.append(V_new.copy())
        V = V_new

        if delta < tol:
            break

    return V, history

- 价值迭代 (`value_iteration`)：
    - 从 V=0 出发，用 Bellman 最优方程更新：`V(s) ← max_a [R(s,a) + γΣ P·V]`
    - 收敛后提取最优策略 `π*(s) = argmax_a Q*(s,a)`

In [ ]:
def value_iteration(max_iter=1000, tol=1e-8):

    V = np.zeros(N_STATES)
    history = [V.copy()]

    for iteration in range(max_iter):
        V_new = np.zeros(N_STATES)

        for s in range(N_STATES):
            q_values = []
            for a in range(N_ACTIONS):
                q = R[s][a]
                for next_s, prob in P[s][a].items():
                    q += GAMMA * prob * V[next_s]
                q_values.append(q)

            V_new[s] = max(q_values)

        delta = np.max(np.abs(V_new - V))
        history.append(V_new.copy())
        V = V_new

        if delta < tol:
            break

    optimal_policy = extract_optimal_policy(V)

    return V, optimal_policy, history

- 验证逻辑 (`verify_results`)：
    - 比较 1：手算 vs. 迭代求解 V^π(s) → 结果一致
    - 比较 2：V^π(s)（随机策略）vs. V*(s)（最优策略）→ V*(s) ≥ V^π(s) 恒成立

In [ ]:
def verify_results():

    uniform_policy = np.ones((N_STATES, N_ACTIONS)) / N_ACTIONS

    print("=" * 60)
    print("  Numerical verification of the Bellman equations")
    print("=" * 60)
    print()

    print("-" * 60)
    print("  Comparison 1: two ways of solving the Bellman expectation equation V^π(s)")
    print("-" * 60)

    manual_V = manual_bellman_expectation()

    iter_V, iter_history = policy_evaluation(uniform_policy)
    print("Policy evaluation (iterative solution) result:")
    for i in range(N_STATES):
        print(f"  V^π(s{i}) = {iter_V[i]:.6f}")
    print()

    print(">>> Comparison result:")
    print(f"  {'State':<8s} {'Manual':<15s} {'Iterative':<15s} {'Error':<15s}")
    for i in range(N_STATES):
        error = abs(manual_V[i] - iter_V[i])
        print(f"  s{i:<6d} {manual_V[i]:<15.8f} {iter_V[i]:<15.8f} {error:<15.2e}")

    all_match = np.allclose(manual_V, iter_V, atol=1e-6)
    print(f"\n  Conclusion: {'Fully consistent ✓' if all_match else 'Discrepancy found ✗'}")
    print(f"  (Manually solving the linear system = iteratively converging -- different roads, same destination!)")
    print()

    print("-" * 60)
    print("  Comparison 2: V^π(s) vs V*(s)")
    print("-" * 60)
    print()

    V_star, optimal_policy, vi_history = value_iteration()

    print("Bellman expectation equation V^π(s) (under the uniform random policy):")
    for i in range(N_STATES):
        print(f"  V^π(s{i}) = {iter_V[i]:.6f}")

    print()
    print("Bellman optimality equation V*(s) (under the optimal policy):")
    for i in range(N_STATES):
        print(f"  V*(s{i}) = {V_star[i]:.6f}")

    print()
    print("Optimal policy π*:")
    action_names = ['a0', 'a1']
    for s in range(N_STATES):
        best = np.argmax(optimal_policy[s])
        print(f"  π*(s{s}) = {action_names[best]}")

    print()
    print(f"  {'State':<8s} {'V^π(s) random policy':<20s} {'V*(s) optimal policy':<20s} {'Gain':<10s}")
    for i in range(N_STATES):
        improvement = V_star[i] - iter_V[i]
        print(f"  s{i:<6d} {iter_V[i]:<20.6f} {V_star[i]:<20.6f} {improvement:>+10.6f}")

    print()
    print("  Analysis: V*(s) >= V^π(s) holds for all states (the optimal policy is never worse)")

    visualize_convergence(iter_history, vi_history)

#### GridWorld 价值迭代 + Q-Learning

[Executable Code](./gridworld_q_learning.py)

目的：在经典 4×4 网格世界中对比规划（value iteration）与学习（Q-Learning）

核心设计：

- 环境：4×4 网格，起点 (0,0)，陷阱 (1,1)，目标 (3,3)，步奖励 -0.01，进入目标 +1.0，进入陷阱 -1.0，γ=0.95。`transition()` 函数实现确定性转移。

In [ ]:
GRID_SIZE = 4
START: State = (0, 0)
TRAP: State = (1, 1)
GOAL: State = (3, 3)
TERMINALS = {TRAP, GOAL}

ACTIONS: tuple[tuple[str, State], ...] = (
    ("up", (-1, 0)),
    ("down", (1, 0)),
    ("left", (0, -1)),
    ("right", (0, 1)),
)
ARROWS = ("↑", "↓", "←", "→")

STEP_REWARD = -0.01
TRAP_REWARD = -1.0
GOAL_REWARD = 1.0
GAMMA = 0.95
ALPHA = 0.15
EPISODES = 500
N_SEEDS = 30
MAX_STEPS = 100


def all_states() -> list[State]:
    return [(row, col) for row in range(GRID_SIZE) for col in range(GRID_SIZE)]


def transition(state: State, action: int) -> tuple[State, float, bool]:

    if state in TERMINALS:
        return state, 0.0, True

    dr, dc = ACTIONS[action][1]
    next_state = (state[0] + dr, state[1] + dc)
    if not (0 <= next_state[0] < GRID_SIZE and 0 <= next_state[1] < GRID_SIZE):
        next_state = state

    if next_state == GOAL:
        return next_state, GOAL_REWARD, True
    if next_state == TRAP:
        return next_state, TRAP_REWARD, True
    return next_state, STEP_REWARD, False


- 价值迭代 (`value_iteration`)：
    - 标准同步 `VI：V(s) ← max_a [R + γ·V(s')]`
    - 收敛容差 1e-12，实际 7 轮收敛
    - 验证：`V*(0,0) ≈ 0.728537125`

In [ ]:
def value_iteration(tolerance: float = 1e-12) -> tuple[dict[State, float], list[dict[State, float]]]:
    """Synchronous value iteration; history[0] is the all-zero initialization."""

    values = {state: 0.0 for state in all_states()}
    history = [values.copy()]

    for _ in range(1_000):
        updated = values.copy()
        for state in all_states():
            if state not in TERMINALS:
                updated[state] = max(
                    action_value(values, state, action)
                    for action in range(len(ACTIONS))
                )
        delta = max(abs(updated[state] - values[state]) for state in all_states())
        values = updated
        history.append(values.copy())
        if delta < tolerance:
            break
    else:
        raise RuntimeError("Value iteration did not converge within 1000 sweeps")

    return values, history

- Q-Learning (`train_q_learning`)：
    - 核心更新：`Q(s,a) += α · [R + γ·max_a' Q(s',a') - Q(s,a)]` (TD(0))
    - α=0.15，ε 按策略衰减
    - 三种 ε 策略对比：线性衰减 (1.0→0.05)、固定 0.05、固定 0.30

In [ ]:
def train_q_learning(seed: int, epsilon_schedule: Callable[[int], float]) -> TrainingRun:
    rng = random.Random(seed)
    q_values = zero_q_table()
    rewards: list[float] = []
    steps_per_episode: list[int] = []

    for episode in range(EPISODES):
        state = START
        episode_reward = 0.0
        epsilon = epsilon_schedule(episode)

        for step in range(1, MAX_STEPS + 1):
            if rng.random() < epsilon:
                action = rng.randrange(len(ACTIONS))
            else:
                action = rng.choice(best_action_indices(q_values, state))

            next_state, reward, done = transition(state, action)
            next_best = 0.0 if done else max(q_values[next_state[0]][next_state[1]])
            old_value = q_values[state[0]][state[1]][action]
            td_target = reward + GAMMA * next_best
            q_values[state[0]][state[1]][action] += ALPHA * (td_target - old_value)

            episode_reward += reward
            state = next_state
            if done:
                break

        rewards.append(episode_reward)
        steps_per_episode.append(step)

    return TrainingRun(rewards, steps_per_episode, q_values)

- 多种子统计：每个 ε 策略跑 30 个种子，取均值和标准差，确保结果可靠

In [ ]:
def aggregate_runs(runs: list[TrainingRun]) -> dict[str, list[float]]:
    reward_mean = [fmean(run.rewards[i] for run in runs) for i in range(EPISODES)]
    reward_std = [pstdev(run.rewards[i] for run in runs) for i in range(EPISODES)]
    step_mean = [fmean(run.steps[i] for run in runs) for i in range(EPISODES)]
    return {"reward_mean": reward_mean, "reward_std": reward_std, "step_mean": step_mean}


def moving_average(values: list[float], window: int = 20) -> list[float]:
    result = []
    for index in range(len(values)):
        left = max(0, index - window + 1)
        result.append(fmean(values[left:index + 1]))
    return result


def run_schedule(name: str, schedule: Callable[[int], float]) -> dict[str, object]:
    runs = [train_q_learning(seed, schedule) for seed in range(N_SEEDS)]
    aggregate = aggregate_runs(runs)
    evaluations = [greedy_evaluation(run.q_values) for run in runs]
    return {
        "name": name,
        "runs": runs,
        "aggregate": aggregate,
        "last_100_reward": fmean(
            reward for run in runs for reward in run.rewards[-100:]
        ),
        "success_rate": fmean(result["success"] for result in evaluations),
        "evaluation_steps": fmean(result["steps"] for result in evaluations),
        "evaluation_reward": fmean(result["reward"] for result in evaluations),
    }

- 纯 SVG 可视化：不依赖 matplotlib，手写 SVG 绘制：
    - 环境图（起点/陷阱/目标标记）
    - 价值迭代传播过程（多轮快照）
    - Q-Learning 学习曲线

#### 整体架构总结

|文件|算法|状态空间|核心公式|
|---|---|---|---|
|`two_armed_bandit.py`|ε-greedy, UCB|1 状态, 2 动作|Q(a) += (R-Q)/N|
|`bellman_equation_verify.py`|策略评估, 价值迭代|3 状态, 2 动作|V←Σπ[R+γPV], V←max[R+γPV]|
|`gridworld_q_learning.py`|价值迭代, Q-Learning|16 状态, 4 动作|V←max_a Q(s,a), Q←α(R+γmaxQ-Q)|

**从最简单的单状态决策问题（bandit），到完整 MDP 的规划与求解（Bellman 验证），再到网格世界中无模型学习（Q-Learning）。**